In [1]:
!git clone https://github.com/HenriqueSchmitz/mario-the-explorer

Cloning into 'mario-the-explorer'...
remote: Enumerating objects: 155, done.
remote: Counting objects: 100% (155/155), done.
remote: Compressing objects: 100% (96/96), done.
remote: Total 155 (delta 63), reused 123 (delta 38), pack-reused 0 (from 0)
Receiving objects: 100% (155/155), 770.16 KiB | 3.57 MiB/s, done.
Resolving deltas: 100% (63/63), done.


In [2]:
!sh ./mario-the-explorer/setup.sh

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 156.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.1/123.1 MB 160.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 157.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 220.2 MB/s eta 0:00:00
Importing SuperMarioWorld-Snes-v0
Imported 1 games


In [3]:
from typing import Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from mario_the_explorer import SuperMarioWorldEmulator, RewardModel, ScreenOverlay, Tile, get_file_logger

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import os

class TileEncoderNet(nn.Module):

    def __init__(self, num_ids, embedding_dim, output_dim, rows, cols, **kwargs):
        super().__init__()
        self.embedding = nn.Embedding(num_ids, embedding_dim)
        self.pos_embedding = nn.Parameter(torch.randn(1, embedding_dim, rows, cols))

        self.spatial_conv = nn.Sequential(
            nn.Conv2d(embedding_dim, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, output_dim, kernel_size=3, padding=1),
            nn.ReLU()
        )

        self.reconstruction_head = nn.Conv2d(output_dim, num_ids, kernel_size=1)

    def forward(self, x, return_features=False):
        is_batch = x.ndim == 3
        if not is_batch:
            x = x.unsqueeze(0)
        x = self.embedding(x).permute(0, 3, 1, 2)
        x = x + self.pos_embedding
        features = self.spatial_conv(x)

        if return_features:
            return features.squeeze(0) if not is_batch else features

        logits = self.reconstruction_head(features)
        return logits.squeeze(0) if not is_batch else logits


class TileEncoder:

    def __init__(self, num_ids=65536, embedding_dim=8, output_dim=16, rows=14, cols=16, learning_rate=1e-3, device=None):
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.params = {
            'num_ids': num_ids,
            'embedding_dim': embedding_dim,
            'output_dim': output_dim,
            'rows': rows,
            'cols': cols,
            'learning_rate': learning_rate}

        self.model = TileEncoderNet(**self.params).to(self.device)
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=learning_rate)
        self.criterion = nn.CrossEntropyLoss()

    def embed(self, observation):
        self.model.eval()
        with torch.no_grad():
            obs_tensor = self._ensure_tensor(observation).to(self.device)
            return self.model(obs_tensor, return_features=True).permute(1, 2, 0)

    def train(self, observations, epochs=5, batch_size=32, mask_prob=0.15):
        if not observations: return

        loader = DataLoader(self._prepare_dataset(observations), batch_size=batch_size, shuffle=True)
        self.model.train()

        for epoch in range(epochs):
            self._run_epoch(loader, mask_prob)

    def save(self, path):
        checkpoint = {
            'model_state': self.model.state_dict(),
            'params': self.params
        }
        torch.save(checkpoint, path)

    @staticmethod
    def load(path, device=None):
        device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        checkpoint = torch.load(path, map_location=device)

        instance = TileEncoder(**checkpoint['params'], device=device)
        instance.model.load_state_dict(checkpoint['model_state'])
        return instance

    def _run_epoch(self, loader, mask_prob):
        for batch in loader:
            batch = batch.to(self.device)
            masked_input = self._apply_mask(batch, mask_prob)

            self.optimizer.zero_grad()
            output = self.model(masked_input)

            loss = self.criterion(output, batch)
            loss.backward()
            self.optimizer.step()

    def _apply_mask(self, batch, mask_prob):
        mask = (torch.rand(batch.shape, device=self.device) < mask_prob)
        masked_batch = batch.clone()
        masked_batch[mask] = 0
        return masked_batch

    def _ensure_tensor(self, data):
        return data if isinstance(data, torch.Tensor) else torch.LongTensor(data)

    def _prepare_dataset(self, observations):
        class SimpleDataset(Dataset):
            def __init__(self, data): self.data = [torch.LongTensor(o) for o in data]
            def __len__(self): return len(self.data)
            def __getitem__(self, i): return self.data[i]
        return SimpleDataset(observations)

In [6]:
class NoRewardModel(RewardModel):

    def get_reward(self,
                   action: list[int],
                   observation: list[list[Tile]],
                   terminated: bool,
                   truncated: bool,
                   info: dict) -> float:
        return 0.0

In [11]:
WALK_RIGHT_ACTION = [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]

In [12]:
RUN_NAME = "data_collection"
LEVEL = "DonutPlains1"
LOG_LEVEL = "INFO"

In [13]:
from gymnasium.wrappers import RecordVideo


logger = get_file_logger(RUN_NAME, LOG_LEVEL)
env = SuperMarioWorldEmulator(level = LEVEL,
                              render_mode = "rgb_array",
                              reward_model = NoRewardModel(),
                              logger = logger)
try:
    distinct_observations = []
    last_distinct_obs = None
    obs = env.reset()
    done = False
    while not done:
        env.render()
        obs, reward, terminated, truncated, info = env.step(WALK_RIGHT_ACTION)
        if last_distinct_obs is None or not np.array_equal(obs, last_distinct_obs):
            last_distinct_obs = obs
            distinct_observations.append(obs)
        done = terminated or truncated
        if done:
            logger.info(f"Terminated: {terminated}")
            logger.info(f"Truncated: {truncated}")
except Exception as e:
    logger.error(e)
    raise e
finally:
    env.close()

2026-04-29 17:13:44 [INFO] Session log for run data_collection with level [INFO] initialized at: data_collection_20260429_171344.log
2026-04-29 17:13:51 [INFO] Terminated: True
2026-04-29 17:13:51 [INFO] Truncated: False


In [14]:
encoder = TileEncoder()

In [15]:
encoder.train(distinct_observations)

In [16]:
example_img = torch.tensor(distinct_observations[-1], device=device)
example_img

tensor([[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0],
        [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0],
        [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0],
        [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0],
        [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0],
        [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0, 811, 811, 811,
         811,   0],
        [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0],
        [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0],
        [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0, 371,   0,
           0,   0],
        [  0,   0,   0,   0,   0,   0,   0,   0, 769, 768, 768, 768, 768, 768,
         768, 768],
        [ 

In [19]:
result = encoder.embed(example_img)
result.shape

torch.Size([14, 16, 16])

In [18]:
result

tensor([[[ 0.0000,  0.0000,  4.4394,  ...,  3.7473,  3.2579,  0.0000],
         [ 0.0000,  0.0000,  7.1642,  ...,  7.2024,  6.0264,  0.0000],
         [ 0.0000,  0.0000,  8.7130,  ...,  8.8268,  6.8277,  0.0000],
         ...,
         [ 0.0000,  0.0000,  7.1986,  ...,  6.6843,  5.6267,  0.0000],
         [ 0.0000,  0.0000,  6.7129,  ...,  6.5322,  5.5821,  0.0000],
         [ 0.0000,  0.0000,  4.3789,  ...,  4.6653,  3.4041,  0.0000]],

        [[ 0.0000,  0.0000,  5.4699,  ...,  5.1945,  4.6622,  0.0000],
         [ 0.0000,  0.0000,  9.2860,  ...,  9.5293,  9.1779,  0.0000],
         [ 0.0000,  0.0000, 11.3209,  ..., 11.3039, 10.4836,  0.0000],
         ...,
         [ 0.0000,  0.0000,  9.9631,  ..., 10.4233,  8.5938,  0.0000],
         [ 0.0000,  0.0000,  9.5061,  ..., 10.3763,  8.6598,  0.0000],
         [ 0.0000,  0.0000,  5.7119,  ...,  7.4107,  5.4134,  0.0000]],

        [[ 0.0000,  0.0000,  6.6619,  ...,  6.0202,  5.0100,  0.0000],
         [ 0.0000,  0.0000, 10.0215,  ..., 10